# Hospital Readmission Prediction #


This will predict 30-day readmission with three models and check the best model works equally well for different age ,sex and race groups.

Data: the cleaned dataset from my Project_readmission_eda (68,167 patients, one visit each).

In [1]:
import numpy as np # linear algebra
import pandas as pd # data processing, CSV file I/O (e.g. pd.read_csv)
import matplotlib.pyplot as plt
import seaborn as sns

sns.set_theme(style="whitegrid")
pd.set_option("display.max_columns", None)

import os
for dirname, _, filenames in os.walk('/kaggle/input'):
    for filename in filenames:
        print(os.path.join(dirname, filename))

/kaggle/input/datasets/manishasingh07/diabetes-clean/diabetes_clean.csv


In [2]:
df= pd.read_csv('/kaggle/input/datasets/manishasingh07/diabetes-clean/diabetes_clean.csv')

In [3]:
df.shape

(68167, 48)

In [4]:
df = df.drop(columns=["readmitted"])
df.shape

(68167, 47)

Dropped the original `readmitted` column because `readmitted_30` was created from it. Keeping it would leak the answer to the model (data leakage).

In [5]:
df["readmitted_30"].value_counts(normalize=True)*100

readmitted_30
0    90.964836
1     9.035164
Name: proportion, dtype: float64

9.04% of patients were readmitted within 30 days and 90.96% were not. The classes are imbalanced, so accuracy would be misleading: a model that always says "no" would be 91% accurate but catch no one.

In [6]:
df["total_visits"] = df["number_outpatient"] + df["number_emergency"] + df["number_inpatient"]
df["total_visits"].describe()

count    68167.00000
mean         0.56853
std          1.43984
min          0.00000
25%          0.00000
50%          0.00000
75%          1.00000
max         49.00000
Name: total_visits, dtype: float64

New feature `total_visits` = outpatient + emergency + inpatient visits in the previous year. Most patients had 0 (median 0), but a few had up to 49. Past hospital use was the strongest finding in Project 1.

In [7]:
med_cols = ["metformin", "repaglinide", "nateglinide", "chlorpropamide", "glimepiride",
            "acetohexamide", "glipizide", "glyburide", "tolbutamide", "pioglitazone",
            "rosiglitazone", "acarbose", "miglitol", "troglitazone", "tolazamide",
            "examide", "citoglipton", "insulin", "glyburide-metformin",
            "glipizide-metformin", "glimepiride-pioglitazone",
            "metformin-rosiglitazone", "metformin-pioglitazone"]

df["meds_changed"] = df[med_cols].isin(["Up", "Down"]).sum(axis=1)
df["meds_changed"].value_counts()

meds_changed
0    51340
1    15847
2      904
3       73
4        3
Name: count, dtype: int64

New feature `meds_changed` = number of diabetes medicines whose dose went Up or Down. 75% of patients had no changes, 23% had one, and about 1.4% had two or more.

In [8]:
groups = df[["age", "gender", "race"]].copy()

save original age, gender and race before encoding, for the fairness check later

In [9]:
df["age"] = df["age"].str[1:].str.split("-").str[0].astype(int) // 10

In [10]:
for col in ["medical_specialty", "diag_1", "diag_2", "diag_3"]:
    top10 = df[col].value_counts().index[:10]
    df[col] = df[col].where(df[col].isin(top10), "Other")

In [11]:
for col in ["admission_type_id", "discharge_disposition_id", "admission_source_id"]:
    df[col] = df[col].astype(str)

 keep the 10 most common values in these columns and group the rest as "Other", to avoid hundreds of rare columns

In [12]:
cat_cols = df.select_dtypes(include="object").columns
df = pd.get_dummies(df, columns=cat_cols, drop_first=True, dtype=int)
df.shape

(68167, 152)

In [13]:
print("total_visits" in df.columns, "meds_changed" in df.columns)
print([c for c in df.columns if "readmit" in c.lower()])

True True
['readmitted_30']


Saved the original age, gender and race for the fairness check. Converted age groups to numbers (0–9). For medical_specialty and diag_1/2/3, kept the 10 most common values and grouped the rest as "Other" to avoid hundreds of rare columns. Treated the admission and discharge ID codes as categories, then one-hot encoded all text columns so every feature is a number.

In [14]:
# y = the answer, X = the patient information, patient_ids = kept aside for checking the split
y = df["readmitted_30"]
patient_ids = df["patient_nbr"]
X = df.drop(columns=["readmitted_30", "patient_nbr"])
X.shape, y.shape

((68167, 150), (68167,))

In [15]:
from sklearn.model_selection import train_test_split

X_train, X_test, y_train, y_test, id_train, id_test, g_train, g_test = train_test_split(
    X, y, patient_ids, groups,
    test_size=0.2, stratify=y, random_state=42
)
X_train.shape, X_test.shape

((54533, 150), (13634, 150))

Split 80/20 with stratification: 54,533 training and 13,634 test patients. 0 patients appear in both sets (each patient has only one visit after Project 1 cleaning). Both sets have about 9% readmitted.

In [16]:
print(df.shape)
print("Leak check:", "readmitted_30" in X.columns, "patient_nbr" in X.columns)
print("Patients in both:", len(set(id_train) & set(id_test)))
print("Readmitted % train:", round(y_train.mean()*100, 2), "| test:", round(y_test.mean()*100, 2))

(68167, 152)
Leak check: False False
Patients in both: 0
Readmitted % train: 9.03 | test: 9.04


In [17]:
from sklearn.pipeline import make_pipeline
from sklearn.preprocessing import StandardScaler
from sklearn.linear_model import LogisticRegression
from sklearn.ensemble import RandomForestClassifier, HistGradientBoostingClassifier

models = {
    "Logistic regression": make_pipeline(StandardScaler(),
                                         LogisticRegression(class_weight="balanced", max_iter=1000)),
    "Random forest": RandomForestClassifier(n_estimators=300, min_samples_leaf=5,
                                            class_weight="balanced", n_jobs=-1, random_state=42),
    "Gradient boosting": HistGradientBoostingClassifier(class_weight="balanced", random_state=42),
}

for name, model in models.items():
    model.fit(X_train, y_train)
    print(name, "trained")

Logistic regression trained
Random forest trained
Gradient boosting trained


In [18]:
from sklearn.metrics import roc_auc_score, average_precision_score, recall_score

rows = []
for name, model in models.items():
    proba = model.predict_proba(X_test)[:, 1]
    pred = model.predict(X_test)
    rows.append({"Model": name,
                 "ROC-AUC": round(roc_auc_score(y_test, proba), 3),
                 "PR-AUC": round(average_precision_score(y_test, proba), 3),
                 "Recall": round(recall_score(y_test, pred), 3)})
results = pd.DataFrame(rows)
results

,Model,ROC-AUC,PR-AUC,Recall
0,Logistic regression,0.650,0.170,0.538
1,Random forest,0.643,0.170,0.108
2,Gradient boosting,0.652,0.177,0.514


All three models perform similarly (ROC-AUC about 0.65). Gradient boosting is slightly best (ROC-AUC 0.652, PR-AUC 0.177). Logistic regression is almost as good and much simpler to explain. Random forest ranks patients well, but at the default 0.5 threshold its recall is only 0.108, so it misses most readmitted patients.

Which metric matters most? With only 9% readmitted, PR-AUC is the most informative, because it measures how well the model finds the rare readmitted patients. The baseline is 0.09, so 0.177 is about twice as good as guessing. Recall also matters in healthcare, because missing a patient who comes back is worse than a false alarm.

In [19]:
from sklearn.inspection import permutation_importance

best_name = results.sort_values("PR-AUC", ascending=False)["Model"].iloc[0]
best_model = models[best_name]
print("Best model:", best_name)

sample = X_test.sample(5000, random_state=42)
imp = permutation_importance(best_model, sample, y_test.loc[sample.index],
                             scoring="roc_auc", n_repeats=3, random_state=42, n_jobs=-1)
importance = pd.Series(imp.importances_mean, index=X_test.columns).sort_values(ascending=False)
importance.head(10)

Best model: Gradient boosting


discharge_disposition_id_3     0.026536
discharge_disposition_id_22    0.016747
number_inpatient               0.013484
time_in_hospital               0.010668
discharge_disposition_id_5     0.009396
number_diagnoses               0.006453
age                            0.006017
num_lab_procedures             0.004496
admission_type_id_6            0.004040
total_visits                   0.003607
dtype: float64

In [20]:
pred = best_model.predict(X_test)
err = X_test[["age", "number_inpatient", "total_visits", "time_in_hospital",
              "num_medications", "number_diagnoses"]].copy()
err["actual"] = y_test.values
err["pred"] = pred
err["type"] = np.select(
    [(err.actual == 1) & (err.pred == 1), (err.actual == 1) & (err.pred == 0),
     (err.actual == 0) & (err.pred == 1)],
    ["Caught (TP)", "Missed (FN)", "False alarm (FP)"], default="Correct no (TN)")
print(err["type"].value_counts())
err.groupby("type").mean().round(2)

type
Correct no (TN)     8725
False alarm (FP)    3677
Caught (TP)          633
Missed (FN)          599
Name: count, dtype: int64


,age,number_inpatient,total_visits,time_in_hospital,num_medications,number_diagnoses,actual,pred
type,,,,,,,,
Caught (TP),6.56,0.65,1.19,5.55,18.04,8.02,1.0,1.0
Correct no (TN),5.80,0.02,0.32,3.56,14.44,6.87,0.0,0.0
False alarm (FP),6.57,0.48,1.05,5.78,18.10,8.03,0.0,1.0
Missed (FN),6.05,0.03,0.40,3.71,14.87,7.13,1.0,0.0


The model mostly catches patients with past hospital stays (0.65 on average), longer stays (5.6 days) and more medicines. The patients it misses had almost no past stays (0.03) and shorter stays (3.7 days), so they looked low-risk in the data, almost identical to patients who correctly did not return. The model has no clues for why they came back. False alarms had the same "sick" profile as caught patients but did not return.

In [21]:
proba = best_model.predict_proba(X_test)[:, 1]
rows = []
for col in ["age", "gender", "race"]:
    for grp in sorted(g_test[col].unique()):
        mask = (g_test[col] == grp).values
        yt = y_test[mask]
        if mask.sum() < 100 or yt.nunique() < 2:
            continue
        rows.append({"Attribute": col, "Group": grp, "Patients": int(mask.sum()),
                     "Readmitted %": round(yt.mean()*100, 1),
                     "ROC-AUC": round(roc_auc_score(yt, proba[mask]), 3),
                     "Recall": round(recall_score(yt, pred[mask]), 3)})
fairness = pd.DataFrame(rows)
fairness

,Attribute,Group,Patients,Readmitted %,ROC-AUC,Recall
0,age,[10-20),108,6.5,0.686,0.429
1,age,[20-30),222,5.4,0.675,0.417
2,age,[30-40),528,8.0,0.696,0.310
3,age,[40-50),1322,7.2,0.638,0.379
4,age,[50-60),2362,6.9,0.651,0.380
5,age,[60-70),3036,8.9,0.641,0.456
6,age,[70-80),3470,10.8,0.653,0.612
7,age,[80-90),2205,10.4,0.619,0.659
8,age,[90-100),349,10.9,0.466,0.263
9,gender,Female,7363,9.0,0.657,0.532


Gender: performance is similar for women and men (ROC-AUC 0.657 vs 0.646, recall 0.53 vs 0.49).
Age: the model catches older patients much better (recall 0.61–0.66 for ages 70–90) than younger ones (0.31–0.43 for ages 30–60), probably because age and past visits raise the risk score. The 90–100 group scores poorly (ROC-AUC 0.466), but it has only 349 patients, so it is unreliable.
Race: recall is lower for African American patients (0.418) than Caucasian patients (0.535), a gap of about 12 points in a large group. Asian and Hispanic results look better, but those groups are very small (103 and 316 patients), so the numbers are not reliable.
Data or model? Likely both: Caucasian patients are about 77% of the data, so the model learns mostly their patterns, and small groups give unstable scores. The model also relies heavily on age and past visits.

## A mistake I found and fixed
My first results showed ROC-AUC = 1.0 for every group, which is unrealistic for readmission. I found that the original `readmitted` column had been one-hot encoded into the features (readmitted_NO, readmitted_>30), so the model could read the answer. After dropping it before encoding, the scores became realistic (about 0.65).